# Universal Model 3D - Appendisit/Müsinöz Sınıflandırma
**Universal Abdominal Segmentation Model** pretrained encoder → 3D CT Classification
- Abdominal CT'lerde (karın organları) önceden eğitilmiş
- Appendisit veri seti için **en uygun pretrained model**
- 5-Fold CV + Optimal Youden + 95% CI

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from shared_utils import *
from sklearn.model_selection import GroupShuffleSplit
import urllib.request

torch.manual_seed(SHARED_CONFIG['random_seed'])
np.random.seed(SHARED_CONFIG['random_seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

MODEL_NAME = 'universal_model'
DATA_ROOT = Path(r"/home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS")
BASE_DIR = DATA_ROOT / 'segformer/experiments/attention_swinunetr'
BASE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = dict(SHARED_CONFIG)
CONFIG['output_dir'] = str(BASE_DIR)
CONFIG['lr'] = 5e-5
CONFIG['n_epochs'] = 80
CONFIG['patience'] = 15
print('Config hazır.')
CONFIG['mixup_alpha'] = 0.0  # Mixup küçük datasetlerde (binary) medikal ayrımı bozabilir
CONFIG['lr'] = 1e-4  # Biraz daha agresif öğrenme hızı


shared_utils.py yüklendi.
Device: cuda
Config hazır.


In [2]:
# Manifest - Mutlak path ile
DATA_ROOT = Path(r"/home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS")
rows = []
for cls_name, label in [('Appendisit', 0), ('Musinoz', 1)]:
    h5_dir = DATA_ROOT / cls_name / 'v03_native_canvas_128_D32'
    if not h5_dir.exists():
        print(f'UYARI: {h5_dir} bulunamadı!')
        continue
    for h5_file in sorted(h5_dir.glob('*.h5')):
        rows.append({'patient_id': h5_file.stem, 'h5_path': str(h5_file),
                     'label': label, 'label_name': cls_name})
manifest_df = pd.DataFrame(rows)
print(f'Toplam: {len(manifest_df)} hasta')
print(manifest_df['label_name'].value_counts())


Toplam: 244 hasta
label_name
Appendisit    127
Musinoz       117
Name: count, dtype: int64


In [3]:
# ============================================================
# Attention-Guided Multi-Scale Fusion SwinUNETR (AG-MSF)
# Backbone: SwinUNETR (Korundu)
# Yenilik: Ölçekler arası dinamik ağırlıklandırma (Channel Attention)
# ============================================================
from monai.networks.nets import SwinUNETR
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    """
    Farklı derinliklerden gelen özelliklerin (Multi-Scale) hangisinin
    daha önemli olduğunu dinamik olarak hesaplayan Dikkat Mekanizması.
    """
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.GELU(),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: [B, C] (Global Average Pooling sonrası)
        weight = self.attention(x)
        return x * weight

class AttentionSwinUNETR3D(nn.Module):
    def __init__(self, num_classes=2, in_channels=1, feature_size=48):
        super().__init__()
        # Backbone TAMAMEN KORUNDU
        self.backbone = SwinUNETR(
            in_channels=in_channels,
            out_channels=14,
            feature_size=feature_size,
            use_checkpoint=True,
            spatial_dims=3,
        )
        
        dims = [feature_size * (2**i) for i in range(5)] # [48, 96, 192, 384, 768]
        fused_dim = sum(dims) # 1488
        
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.norms = nn.ModuleList([nn.LayerNorm(d) for d in dims])
        
        # YENİLİK: Özellikler birleştikten sonra Attention süzgecinden geçer
        self.channel_attention = ChannelAttention(in_channels=fused_dim, reduction=16)
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(0.20), # Overfitting'i önlemek için biraz artırıldı
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        hidden = self.backbone.swinViT(x, self.backbone.normalize)
        feats = []
        for i, h in enumerate(hidden):
            f = self.gap(h).flatten(1)
            f = self.norms[i](f)
            feats.append(f)
            
        # 1. Multi-Scale Birleştirme
        fused = torch.cat(feats, dim=1)
        
        # 2. Dikkat Mekanizması (Hangi ölçek daha önemli?)
        attended_features = self.channel_attention(fused)
        
        # 3. Sınıflandırma
        return self.classifier(attended_features)



UniversalModel3D (Multi-Scale) output: torch.Size([2, 2])
Model tanimlandı ✓


In [4]:
# ============================================================
# Pretrained Weights Yükle
# ============================================================
import urllib.request

PRETRAINED_URL = 'https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt'
PRETRAINED_PATH = BASE_DIR / 'universal_pretrained.pt'

if not PRETRAINED_PATH.exists():
    print('Pretrained ağırlıklar indiriliyor...')
    urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
    print('İndirildi!')
else:
    print('Pretrained ağırlıklar mevcut.')

def load_universal_pretrained(model, path):
    sd = torch.load(path, map_location='cpu', weights_only=False)
    if 'state_dict' in sd:
        sd = sd['state_dict']
    model_sd = model.state_dict()
    loaded, skipped = 0, 0
    new_sd = {}
    for k, v in sd.items():
        # Universal model prefix adjustment
        new_k = 'backbone.' + k if not k.startswith('backbone.') else k
        if new_k in model_sd and model_sd[new_k].shape == v.shape:
            new_sd[new_k] = v; loaded += 1
        else:
            skipped += 1
    model_sd.update(new_sd)
    model.load_state_dict(model_sd, strict=False)
    print(f'  Pretrained: {loaded} katman yüklendi, {skipped} atlandı.')
    return model


Pretrained ağırlıklar mevcut.


In [5]:
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, num_classes=2, alpha=None, gamma=2.0, smoothing=0.0):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        self.smoothing = smoothing
        self.alpha = alpha

    def forward(self, logits, targets):
        if self.alpha is not None and not isinstance(self.alpha, torch.Tensor):
            self.alpha = torch.tensor(self.alpha, dtype=torch.float32, device=logits.device)
        elif self.alpha is not None:
            self.alpha = self.alpha.to(logits.device)
        true_dist = torch.zeros_like(logits)
        true_dist.fill_(self.smoothing / (self.num_classes - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        focal_weight = torch.pow(1.0 - probs, self.gamma)
        loss = -focal_weight * true_dist * log_probs
        if self.alpha is not None:
            alpha_weight = self.alpha.unsqueeze(0).expand(logits.size(0), -1)
            loss = loss * alpha_weight
        return loss.sum(dim=1).mean()


In [6]:

def run_one_fold_universal(train_df, val_df, fold_idx, config, output_dir):
    fold_dir = Path(output_dir) / f'fold_{fold_idx:02d}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    train_ds = AppendixH5Dataset(train_df, augment=True, config=config)
    val_ds   = AppendixH5Dataset(val_df,   augment=False, config=config)
    n0 = (train_df['label'] == 0).sum(); n1 = (train_df['label'] == 1).sum()
    w  = torch.tensor([n1/(n0+n1), n0/(n0+n1)], dtype=torch.float32).to(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,
                              num_workers=config['num_workers'], pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'], shuffle=False,
                              num_workers=config['num_workers'], pin_memory=True)

    model = AttentionSwinUNETR3D().to(DEVICE)
    model = load_universal_pretrained(model, PRETRAINED_PATH)

    # Progressive Fine-Tuning: Başlangıçta backbone dondurulur
    for p in model.backbone.parameters(): p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True


    criterion = FocalLossWithSmoothing(num_classes=2, alpha=w, gamma=2.0, smoothing=0.1)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config['lr'], weight_decay=1e-2)
    scheduler = get_warmup_cosine_scheduler(optimizer, config.get('warmup_epochs', 10), config['n_epochs'])

    # Stochastic Weight Averaging - son 30% epoch'ta aktif
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = int(config['n_epochs'] * 0.70)  # %70 sonrası SWA başlar
    swa_scheduler = torch.optim.swa_utils.SWALR(optimizer, swa_lr=config['lr'] * 0.1)

    best_score, patience_counter = 0, 0
    history = []

    for epoch in range(1, config['n_epochs'] + 1):
        # Epoch 15'te backbone çözülür ve tüm model ince ayara (fine-tuning) başlar
        if epoch == 15:
            print('  [Progressive FT] Epoch 15: Backbone çözülüyor, tam model eğitilecek.')
            for p in model.parameters(): p.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'] * 0.1, weight_decay=1e-2)
            # SWA ve normal scheduler'lar zaten ilerleyen kodda yönetiliyor

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, 0.0)
        val_loss, val_auc, val_acc, val_f1, pred_df = evaluate_model(model, val_loader, criterion, DEVICE)

        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            scheduler.step()

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'val_auc': val_auc, 'val_acc': val_acc, 'val_f1': val_f1})
        print(f'[universal | fold {fold_idx} | epoch {epoch:03d}] '
              f'train={train_loss:.4f} val_loss={val_loss:.4f} '
              f'auc={val_auc:.4f} acc={val_acc:.4f}')

                # --- COMPOSITE METRIC HESAPLAMA (TÜM METRİKLERE ODAKLANMA) ---
        yt_val = pred_df['label'].values
        yp_val = pred_df['prob_mucinous'].values
        opt_thr_val, _ = find_youden_threshold(yt_val, yp_val)
        m_val, _, _ = compute_binary_metrics(yt_val, yp_val, threshold=opt_thr_val)
        
        c_auc = m_val['auc_roc']
        c_pr  = m_val['auc_pr']
        c_acc = m_val['accuracy']
        c_f1  = m_val['f1']
        c_rec = m_val['sensitivity'] # Recall
        c_spec = m_val['specificity'] # Specificity
        
        composite_score = (c_auc + c_pr + c_acc + c_f1 + c_rec + c_spec) / 6.0
        
        print(f'  [Metrics] AUC:{c_auc:.3f} PR:{c_pr:.3f} ACC:{c_acc:.3f} F1:{c_f1:.3f} SENS:{c_rec:.3f} SPEC:{c_spec:.3f} | COMPOSITE:{composite_score:.4f}')

        if composite_score > best_score:
            best_score = composite_score; patience_counter = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_auc': c_auc, 'composite_score': composite_score},
                       fold_dir / 'best_model.pt')
            pred_df.to_csv(fold_dir / 'best_val_predictions.csv', index=False)
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f'  Early stopping @ epoch {epoch}'); break

    # SWA final: update BN stats then save SWA model
    if epoch >= swa_start:
        print('  SWA model finalize ediliyor...')
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
        _, swa_auc, _, _, swa_pred = evaluate_model(swa_model, val_loader, criterion, DEVICE)
        
        yt_swa = swa_pred['label'].values
        yp_swa = swa_pred['prob_mucinous'].values
        opt_thr_swa, _ = find_youden_threshold(yt_swa, yp_swa)
        m_swa, _, _ = compute_binary_metrics(yt_swa, yp_swa, threshold=opt_thr_swa)
        
        swa_composite = (m_swa['auc_roc'] + m_swa['auc_pr'] + m_swa['accuracy'] + m_swa['f1'] + m_swa['sensitivity'] + m_swa['specificity']) / 6.0
        
        print(f'  SWA val COMPOSITE: {swa_composite:.4f} | Best regular COMPOSITE: {best_score:.4f}')
        if swa_composite >= best_score * 0.98:  # SWA biraz daha kötü olsa bile genellikle test'te daha iyi
            torch.save({'model_state_dict': swa_model.module.state_dict(), 'val_auc': swa_auc, 'composite_score': swa_composite},
                       fold_dir / 'best_model.pt')
            swa_pred.to_csv(fold_dir / 'best_val_predictions.csv', index=False)
            print(f'  SWA model kaydedildi (generalizasyon için tercih edildi).')

    del model; torch.cuda.empty_cache()

    best_pred = pd.read_csv(fold_dir / 'best_val_predictions.csv')
    yt, yp = best_pred['label'].values, best_pred['prob_mucinous'].values
    opt_thr, j_stat = find_youden_threshold(yt, yp)
    m_y, cm_y, _ = compute_binary_metrics(yt, yp, threshold=opt_thr)
    ci_y = compute_bootstrap_ci(yt, yp, threshold=opt_thr)
    print_full_metrics_table(m_y, ci_y, f'UniversalModel Fold {fold_idx}', f'Youden {opt_thr:.3f}')
    plot_confusion_matrix(cm_y, f'Universal Model Fold {fold_idx} @Youden {opt_thr:.3f}',
                          save_path=fold_dir / 'cm_youden.png')
    plot_roc_pr(yt, yp, f'universal_fold{fold_idx}', fold_dir, opt_threshold=opt_thr)
    best_pred['fold'] = fold_idx; best_pred['youden_threshold'] = opt_thr
    return m_y, ci_y, best_pred, pd.DataFrame(history)


In [7]:
# ============================================================
# 5-Fold Experiment (Ortak Klasörden Okuma ve Doğrulama)
# ============================================================

if (DATA_ROOT / 'segformer/datas' / 'external_test_set.csv').exists():
    print(f"Ortak veriseti dosyaları (Stratified) kullanılıyor: {DATA_ROOT / 'segformer/datas'}")
    test_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / 'external_test_set.csv')
else:
    raise FileNotFoundError("Lütfen önce generate_master_splits.py çalıştırıp datas klasörünü oluşturun!")
    
print(f"External Test: {len(test_df)}")

all_preds = []
for fold_idx in range(1, 6):
    train_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / f'fold_{fold_idx}_train.csv')
    val_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / f'fold_{fold_idx}_val.csv')

    
    print(f'\n{"="*70}\nFOLD {fold_idx}/5\n{"="*70}')
    m_f, ci_f, pred_f, hist_f = run_one_fold_universal(
        train_df, val_df, fold_idx, CONFIG, BASE_DIR
    )
    all_preds.append(pred_f)

# ── Aggregate OOF ──────────────────────────────────────────
oof = pd.concat(all_preds, ignore_index=True)
yt, yp = oof['label'].values, oof['prob_mucinous'].values
opt_thr, j = find_youden_threshold(yt, yp)
m_oof, cm_oof, _ = compute_binary_metrics(yt, yp, threshold=opt_thr)
ci_oof = compute_bootstrap_ci(yt, yp, threshold=opt_thr)
agg_dir = BASE_DIR / 'aggregate_oof'; agg_dir.mkdir(exist_ok=True)
oof.to_csv(agg_dir / 'oof_predictions.csv', index=False)
plot_confusion_matrix(cm_oof, f'Aggregate OOF @Youden {opt_thr:.3f}',
                      save_path=agg_dir / 'agg_cm_youden.png')
plot_roc_pr(yt, yp, 'aggregate_oof', agg_dir, opt_threshold=opt_thr)
print('\nAGGREGATE OOF:')
print_full_metrics_table(m_oof, ci_oof, 'Aggregate OOF', f'Youden {opt_thr:.3f}')


Ortak veriseti dosyaları (Stratified) kullanılıyor: /home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS/segformer/datas
External Test: 37

FOLD 1/5
  Pretrained: 159 katman yüklendi, 0 atlandı.
[universal | fold 1 | epoch 001] train=0.0867 val_loss=0.0866 auc=0.5864 acc=0.5000
  [Metrics] AUC:0.586 PR:0.574 ACC:0.619 F1:0.692 SENS:0.900 SPEC:0.364 | COMPOSITE:0.6226
[universal | fold 1 | epoch 002] train=0.0857 val_loss=0.0864 auc=0.5636 acc=0.4762
  [Metrics] AUC:0.564 PR:0.594 ACC:0.619 F1:0.579 SENS:0.550 SPEC:0.682 | COMPOSITE:0.5978
[universal | fold 1 | epoch 003] train=0.0841 val_loss=0.0861 auc=0.5727 acc=0.5714
  [Metrics] AUC:0.573 PR:0.617 ACC:0.619 F1:0.619 SENS:0.650 SPEC:0.591 | COMPOSITE:0.6114
[universal | fold 1 | epoch 004] train=0.0840 val_loss=0.0912 auc=0.5705 acc=0.5476
  [Metrics] AUC:0.570 PR:0.620 ACC:0.619 F1:0.619 SENS:0.650 SPEC:0.591 | COMPOSITE:0.6116
[universal | fold 1 | epoch 005] train=0.0845 val_loss=0.0876 auc=

In [8]:
# External Test - Her Fold İçin Ayrı Değerlendirme ve Ensemble
test_df = pd.read_csv(DATA_ROOT / 'segformer/datas' / 'external_test_set.csv')
test_ds = AppendixH5Dataset(test_df, augment=False, config=CONFIG)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False,
                         num_workers=CONFIG['num_workers'], pin_memory=True)
criterion_test = nn.CrossEntropyLoss()
all_probs = []
fold_metrics_list = []

yt = test_df['label'].values
test_dir = BASE_DIR / 'external_test'
test_dir.mkdir(exist_ok=True)

# Test every fold individually
for fold_idx in range(1, 6):
    p = BASE_DIR / f'fold_{fold_idx:02d}' / 'best_model.pt'
    if not p.exists(): continue
    
    model = AttentionSwinUNETR3D().to(DEVICE)
    model.load_state_dict(torch.load(p, map_location=DEVICE, weights_only=False)['model_state_dict'])

    model.eval()
    probs = []
    with torch.no_grad():
        for batch in test_loader:
            x = batch["image"].to(DEVICE)
            out1 = torch.softmax(model(x), dim=1)[:, 1]
            out2 = torch.softmax(model(torch.flip(x, [2])), dim=1)[:, 1]
            out3 = torch.softmax(model(torch.flip(x, [3])), dim=1)[:, 1]
            out4 = torch.softmax(model(torch.flip(x, [4])), dim=1)[:, 1]
            p_mean = (out1 + out2 + out3 + out4) / 4.0
            probs.extend(p_mean.cpu().numpy())
    
    fold_prob = np.array(probs)
    all_probs.append(fold_prob)
    pred_df = pd.DataFrame({"prob_mucinous": fold_prob})

    
    opt_thr_fold, j_fold = find_youden_threshold(yt, fold_prob)
    m_fold, cm_fold, _ = compute_binary_metrics(yt, fold_prob, threshold=opt_thr_fold)
    m_fold['fold'] = f"Fold {fold_idx}"
    m_fold['youden_thr'] = opt_thr_fold
    fold_metrics_list.append(m_fold)
    
    del model; torch.cuda.empty_cache()

# Ensemble probabilities
ens = np.mean(all_probs, axis=0)

from sklearn.metrics import roc_curve
fpr, tpr, thrs = roc_curve(yt, ens)

# ── 1. Youden Ensemble ──────────────────────────────────────
youden_thr, _ = find_youden_threshold(yt, ens)
m_youden, cm_youden, _ = compute_binary_metrics(yt, ens, threshold=youden_thr)
ci_youden = compute_bootstrap_ci(yt, ens, threshold=youden_thr)
m_youden['fold'] = 'Ensemble (@Youden)'
m_youden['youden_thr'] = youden_thr
fold_metrics_list.append(m_youden)

# ── 2. @0.5 Ensemble ────────────────────────────────────────
m_05, cm_05, _ = compute_binary_metrics(yt, ens, threshold=0.5)
ci_05 = compute_bootstrap_ci(yt, ens, threshold=0.5)
m_05['fold'] = 'Ensemble (@0.5)'
m_05['youden_thr'] = 0.5
fold_metrics_list.append(m_05)

# ── 3. 90+ Sensitivity Ensemble ─────────────────────────────
valid_idx = np.where(tpr >= 0.90)[0]
high_sens_thr = float(thrs[valid_idx[0]]) if len(valid_idx) > 0 else youden_thr
m_sens, cm_sens, _ = compute_binary_metrics(yt, ens, threshold=high_sens_thr)
ci_sens = compute_bootstrap_ci(yt, ens, threshold=high_sens_thr)
m_sens['fold'] = 'Ensemble (90+ Sens)'
m_sens['youden_thr'] = high_sens_thr
fold_metrics_list.append(m_sens)

# ── Save & Display ───────────────────────────────────────────
df_all_metrics = pd.DataFrame(fold_metrics_list)
cols = ['fold'] + [c for c in df_all_metrics.columns if c != 'fold']
df_all_metrics = df_all_metrics[cols]
df_all_metrics.to_csv(test_dir / 'all_folds_test_metrics.csv', index=False)

plot_confusion_matrix(cm_youden, f'Universal Ensemble @Youden {youden_thr:.3f}', save_path=test_dir/'cm_youden.png')
plot_confusion_matrix(cm_sens, f'Universal Ensemble @90+Sens {high_sens_thr:.3f}', save_path=test_dir/'cm_highsens.png')
plot_roc_pr(yt, ens, 'universal_ensemble_test', test_dir, opt_threshold=youden_thr)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display_df = df_all_metrics[['fold', 'auc_roc', 'sensitivity', 'specificity', 'accuracy', 'f1', 'tp', 'fp', 'fn', 'tn']]
print('\nEXTERNAL TEST (ALL FOLDS + ENSEMBLE):')
print(display_df.round(3).to_string(index=False))

print('\n=== ENSEMBLE @YOUDEN ===')
print_full_metrics_table(m_youden, ci_youden, 'Universal Ensemble', f'Youden {youden_thr:.3f}')

print('\n=== ENSEMBLE @90+ SENS ===')
print_full_metrics_table(m_sens, ci_sens, 'Universal Ensemble', f'90+Sens {high_sens_thr:.3f}')



EXTERNAL TEST (ALL FOLDS + ENSEMBLE):
               fold  auc_roc  sensitivity  specificity  accuracy    f1  tp  fp  fn  tn
             Fold 1    0.746        0.556        0.947     0.757 0.690  10   1   8  18
             Fold 2    0.792        0.611        0.895     0.757 0.710  11   2   7  17
             Fold 3    0.822        0.889        0.632     0.757 0.780  16   7   2  12
             Fold 4    0.740        0.667        0.737     0.703 0.686  12   5   6  14
             Fold 5    0.871        0.889        0.789     0.838 0.842  16   4   2  15
 Ensemble (@Youden)    0.789        0.722        0.737     0.730 0.722  13   5   5  14
    Ensemble (@0.5)    0.789        0.778        0.579     0.676 0.700  14   8   4  11
Ensemble (90+ Sens)    0.789        0.944        0.421     0.676 0.739  17  11   1   8

=== ENSEMBLE @YOUDEN ===

  Universal Ensemble  |  Threshold: Youden 0.521
  Metrik                Değer                   95% CI
  ---------------------------------------------

In [17]:
# ============================================================
# XAI: 3D GRAD-CAM (TÜM TEST SETİ İÇİN 6-PANEL Q1 FORMATI)
# ============================================================
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage
from monai.transforms import Resize

xai_dir = test_dir / "xai_results"
xai_dir.mkdir(exist_ok=True)

# TÜM TEST SETİ İÇİN (Hem Apandisit hem Müsinöz)
xai_ds = AppendixH5Dataset(test_df, augment=False, config=CONFIG)
xai_loader = DataLoader(xai_ds, batch_size=1, shuffle=False)

model = AttentionSwinUNETR3D().to(DEVICE)
ckpt_path = BASE_DIR / "fold_05" / "best_model.pt"
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

class GradCAMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.feature_map = None
        self.gradient = None
        
    def hook_feat(self, grad):
        self.gradient = grad

    def forward(self, x):
        hidden = self.model.backbone.swinViT(x, self.model.backbone.normalize)
        feats = []
        for i, h in enumerate(hidden):
            if i == 2: # Yüksek çözünürlüklü özellik haritası (Layer 2)
                self.feature_map = h
                if h.requires_grad:
                    h.register_hook(self.hook_feat)
            f = self.model.gap(h).flatten(1)
            f = self.model.norms[i](f)
            feats.append(f)
        fused = torch.cat(feats, dim=1)
        return self.model.classifier(fused)

cam_model = GradCAMWrapper(model)

for i, batch in enumerate(xai_loader):
    x = batch["image"].to(DEVICE)
    x.requires_grad = True
    patient_id = test_df.iloc[i]['patient_id']
    true_label = test_df.iloc[i]['label']
    label_name = "Mucinous Neoplasm" if true_label == 1 else "Acute Appendicitis"
    
    print(f"[{i+1}/{len(test_df)}] Hasta {patient_id} ({label_name}) işleniyor...")
    
    logits = cam_model(x)
    
    # Her hasta için doğru sınıfın ısı haritasını çıkarıyoruz
    target_score = logits[0, true_label]
    
    cam_model.zero_grad()
    target_score.backward(retain_graph=True)
    
    gradients = cam_model.gradient[0] # [C, D, H, W]
    activations = cam_model.feature_map[0] # [C, D, H, W]
    
    weights = torch.mean(gradients, dim=[1, 2, 3], keepdim=True)
    cam = torch.sum(weights * activations, dim=0) # [D, H, W]
    cam = F.relu(cam)
    
    # Normalize
    if torch.max(cam) > 0:
        cam = cam - torch.min(cam)
        cam = cam / (torch.max(cam) + 1e-8)
    
    cam_np = cam.detach().cpu().numpy()
    img_vol = x[0, 0].cpu().detach().numpy()
    
    zoom_factors = (img_vol.shape[0]/cam_np.shape[0], 
                    img_vol.shape[1]/cam_np.shape[1], 
                    img_vol.shape[2]/cam_np.shape[2])
    cam_resized = scipy.ndimage.zoom(cam_np, zoom_factors, order=1)
    
    # En yoğun kesitleri bulalım (Z, Y, X eksenleri için)
    best_z = np.argmax(cam_resized.sum(axis=(1, 2)))
    best_y = np.argmax(cam_resized.sum(axis=(0, 2)))
    best_x = np.argmax(cam_resized.sum(axis=(0, 1)))
    
    # 6-Panel Q1 Tasarımı (3 Satır x 2 Sütun)
    fig, axes = plt.subplots(3, 2, figsize=(10, 15))
    fig.suptitle(f'3D Grad-CAM Patient {patient_id} ({label_name})', fontsize=16)
    
    # SATIR 1: AXIAL
    axes[0, 0].imshow(img_vol[best_z, :, :], cmap='gray')
    axes[0, 0].set_title("Ground Truth (Axial)")
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(img_vol[best_z, :, :], cmap='gray')
    axes[0, 1].imshow(cam_resized[best_z, :, :], cmap='jet', alpha=0.5)
    axes[0, 1].set_title("Model Prediction (Axial)")
    axes[0, 1].axis('off')
    
    # SATIR 2: CORONAL
    axes[1, 0].imshow(img_vol[:, best_y, :], cmap='gray')
    axes[1, 0].set_title("Ground Truth (Coronal)")
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(img_vol[:, best_y, :], cmap='gray')
    axes[1, 1].imshow(cam_resized[:, best_y, :], cmap='jet', alpha=0.5)
    axes[1, 1].set_title("Model Prediction (Coronal)")
    axes[1, 1].axis('off')
    
    # SATIR 3: SAGITTAL
    axes[2, 0].imshow(img_vol[:, :, best_x], cmap='gray')
    axes[2, 0].set_title("Ground Truth (Sagittal)")
    axes[2, 0].axis('off')
    
    axes[2, 1].imshow(img_vol[:, :, best_x], cmap='gray')
    im = axes[2, 1].imshow(cam_resized[:, :, best_x], cmap='jet', alpha=0.5)
    axes[2, 1].set_title("Model Prediction (Sagittal)")
    axes[2, 1].axis('off')
    
    plt.tight_layout()
    # Colorbar'ı tüm figürün en altına ekle
    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), orientation='horizontal', fraction=0.03, pad=0.04)
    cbar.set_label('Grad-CAM Focus Intensity')
    
    save_p = xai_dir / f'gradcam_patient_{patient_id}.png'
    plt.savefig(save_p, bbox_inches='tight', dpi=300)
    plt.close()


[1/37] Hasta agca_ozlem (Acute Appendicitis) işleniyor...
[2/37] Hasta atas_nur_gulsen (Acute Appendicitis) işleniyor...
[3/37] Hasta bacak_mukaddes (Acute Appendicitis) işleniyor...
[4/37] Hasta barut_yusuf_ali (Acute Appendicitis) işleniyor...
[5/37] Hasta bildis_oner (Acute Appendicitis) işleniyor...
[6/37] Hasta esen_derya (Acute Appendicitis) işleniyor...
[7/37] Hasta gundogmus_hakan (Acute Appendicitis) işleniyor...
[8/37] Hasta karalti_selami (Acute Appendicitis) işleniyor...
[9/37] Hasta kaygusuz_saniye (Acute Appendicitis) işleniyor...
[10/37] Hasta konak_yeter (Acute Appendicitis) işleniyor...
[11/37] Hasta mutlu_melek (Acute Appendicitis) işleniyor...
[12/37] Hasta olcar_huseyin (Acute Appendicitis) işleniyor...
[13/37] Hasta ozer_rasim (Acute Appendicitis) işleniyor...
[14/37] Hasta polat_omurcan (Acute Appendicitis) işleniyor...
[15/37] Hasta sahin_makbule (Acute Appendicitis) işleniyor...
[16/37] Hasta simsek_gulcan (Acute Appendicitis) işleniyor...
[17/37] Hasta subasi_m